In [1]:
!pip install google-adk google-cloud-aiplatform --quiet

In [10]:
from google.colab import auth
auth.authenticate_user()

In [11]:
PROJECT_ID = "qwiklabs-gcp-04-a318f6cee29a"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://weather_agent_for_ai_bootcamp"

import os
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"


In [13]:
import logging
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

import vertexai

PROJECT_ID = "qwiklabs-gcp-04-a318f6cee29a"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://weather_agent_for_ai_bootcamp"

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print("Vertex AI initialized successfully!")

Vertex AI initialized successfully!


In [14]:
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

weather_agent = Agent(
    model="gemini-2.0-flash",
    name="weather_agent",
    description="A helpful weather agent.",
    instruction="Use google_search to find weather for any location the user asks about.",
    tools=[google_search],
)


In [15]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()
runner = Runner(agent=weather_agent, app_name="weather_app", session_service=session_service)

session = await session_service.create_session(app_name="weather_app", user_id="test_user")

async for event in runner.run_async(
    user_id="test_user",
    session_id=session.id,
    new_message=types.Content(role="user", parts=[types.Part(text="What's the weather in Chicago today?")])
):
    if event.is_final_response():
        print(event.content.parts[0].text)


The weather in Chicago, Illinois today, March 6, 2026, is heavy rain. The temperature is 49°F (9°C), but it feels like 46°F (8°C). The chance of rain is 91%, and the humidity is around 100%. The forecast for the rest of the day is rain, with a 92% chance of precipitation. Tonight, there will be light rain with a 70% chance of precipitation.


In [16]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(agent=weather_agent, enable_tracing=False)


In [17]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
    app,
    requirements=["google-adk", "google-cloud-aiplatform"],
    display_name="Weather Agent",
)

print("Deployed! Resource name:", remote_agent.resource_name)


INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.12.3', 'google-cloud-aiplatform': '1.135.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk', 'google-cloud-aiplatform', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket weather_agent_for_ai_bootcamp
INFO:vertexai.agent_engines:Wrote to gs://weather_agent_for_ai_bootcamp/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://weather_agent_for_ai_bootcamp/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://weather_agent_for_ai_bootcamp/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/282553873629/loca

Deployed! Resource name: projects/282553873629/locations/us-central1/reasoningEngines/7493680817177100288


In [20]:
remote_session = remote_agent.create_session(user_id="test_user")

# ✅ Plain for loop — stream_query is a sync generator
for event in remote_agent.stream_query(
    user_id="test_user",
    session_id=remote_session["id"],
    message="What's the weather in Springfield, Illinois today?"
):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            if "text" in part:
                print(part["text"])


The weather in Springfield, Illinois today, March 6, 2026, is mostly cloudy with a temperature of 69°F (21°C), but it feels like 76°F (24°C). There is an 8% chance of rain. The forecast for tonight is light rain with a 75% chance of rain.
